## Tutorial 1 - Introduction to PyTorch
PyTorch is a popular library for deep learning. This tutorial is designed for you to get familiar with PyTorch, by the end of which we will create a basic Neural Network, for classifying digits, trained on the MNIST dataset

### Learning Objectives
By the end of this tutorial, you will:
- Understand PyTorch tensors as mathematical objects
- See neural networks as parameterized functions
- Understand what autograd computes and what it hides
- Implement stochastic gradient descent from first principles
- Analyze optimization behavior via learning rates and model width

### Setup and Reproducibility

We begin by importing required packages and fixing random seeds to ensure reproducibility.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from torchvision import datasets

# Fix random seeds for reproducibility
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)

### Loading MNIST as Raw Tensors
PyTorch does not require datasets or dataloaders. At its core, it operates on tensors. We load the MNIST dataset and manually preprocess it.

In [2]:
train_dataset = datasets.MNIST(root="./data", train=True, download=True)
test_dataset = datasets.MNIST(root="./data", train=False, download=True)

X_train, y_train = train_dataset.data, train_dataset.targets.long()
X_test, y_test = test_dataset.data, test_dataset.targets.long()

In [3]:
# Flatten images and normalize pixel values to [0, 1]
X_train = X_train.reshape(X_train.shape[0], -1).float() / 256
X_test = X_test.reshape(X_test.shape[0], -1).float() / 256

print(f"Train images shape: {X_train.shape}")
print(f"Train labels shape: {y_train.shape}")
print(f"Test images shape: {X_test.shape}")
print(f"Test labels shape: {y_test.shape}")

Train images shape: torch.Size([60000, 784])
Train labels shape: torch.Size([60000])
Test images shape: torch.Size([10000, 784])
Test labels shape: torch.Size([10000])


### Exercise 1: Subsampling the Dataset
To reduce computational cost, we will use only 6000 random samples from the training set.

`Important: images and labels must be sampled using the same indices.`

In [ ]:
num_samples = 6000
indices = np.random.choice(len(X_train), size=num_samples, replace=False)

X_train = X_train[indices]
y_train = y_train[indices]

print(X_train.shape, y_train.shape)

#### Let's visualize some images

In [ ]:
num_images = 10
plt.figure(figsize=(10, 1))
for i in range(num_images):
    plt.subplot(1, num_images, i+1)
    plt.imshow(X_train[i].reshape(28, 28), cmap='gray')
    plt.title(y_train[i].item())
    plt.axis('off')
plt.show()

### Neural Networks as Parameterized Functions
A neural network implements a function:
$$
𝑓_\theta: \mathbb{R}^{784} \rightarrow \mathbb{R}^{100} \rightarrow \mathbb{R}^{10}
$$
where 
$\theta$ denotes all learnable parameters.

In [ ]:
class MLP(torch.nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hidden = torch.nn.Linear(input_size, hidden_size)
        self.output = torch.nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = torch.relu(self.hidden(x))
        x = self.output(x)
        return x


model = MLP(input_size=28*28, hidden_size=100, output_size=10)
for name, param in model.named_parameters():
    print(f"{name}: {param.shape}")

#### Forward Pass Before Training

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(X_train[0].unsqueeze(0))
    prediction = torch.argmax(logits, dim=1).item()

plt.imshow(X_train[0].reshape(28, 28), cmap="gray")
print("Untrained prediction:", prediction)

### Loss Functions and Reduction
The cross-entropy loss is defined per sample:
$$
    l_i = -log p(y_i | x_i)
$$
But PyTorch returns a scalar by aggregating across the batch.

### Exercise 2 - Per-Sample Losses

In [ ]:
criterion_no_reduction = torch.nn.CrossEntropyLoss(reduction="none")
criterion_mean = torch.nn.CrossEntropyLoss()

outputs = model(X_train[:32])
losses = criterion_no_reduction(outputs, y_train[:32])
manual_mean = losses.mean()
default_loss = criterion_mean(outputs, y_train[:32])

print("Per-sample losses:", losses)
print("Manual mean:", manual_mean.item())
print("Default loss:", default_loss.item())

### Autograd Computes Gradients of Scalars
- PyTorch computes gradients of scalar-valued functions only.
- `Observe: gradients correspond to parameters, not samples.`

In [ ]:
criterion = torch.nn.CrossEntropyLoss()
outputs = model(X_train[:32])
loss = criterion(outputs, y_train[:32])

loss.backward()

for name, param in model.named_parameters():
    print(f"{name} grad shape: {param.grad.shape}")

### Training using Stochastic Gradient descent

The Stochastic Gradient Descent update rule is as follows:
$$
\theta \leftarrow \theta - \eta \nabla_\theta \mathcal{L}(f_\theta(X_B), y_B)
$$

Where 
- $\theta$ is the set of parameters, 
- $f_\theta$ is the function represented by the Neural Network, 
- $X_B$ and $y_B$ is a batch of input data and labels, sampled randomly from the training dataset and 
- $\mathcal{L}$ is the Loss function

The above update is done for each iteration until convergence. We will not use `torch.optim`

#### Model Evaluation

In [ ]:
def evaluate_model(model, test_data, test_labels, batch_size):
    model.eval()
    correct = 0
    num_samples = len(test_labels)


    with torch.no_grad():
        for i in range(0, num_samples, batch_size):
            x_batch = test_data[i:i+batch_size]
            y_batch = test_labels[i:i+batch_size]
        
            outputs = model(x_batch)
            predictions = torch.argmax(outputs, dim=1)

            correct += (predictions == y_batch).sum().item()

    return correct / num_samples

#### Training Loop (Core Exercise)

In [ ]:
def train_model(model, train_data, labels, batch_size, learning_rate, num_iterations):
    criterion = torch.nn.CrossEntropyLoss()
    num_samples = len(labels)
    
    losses = []
    test_accuracies = []

    for itr in range(num_iterations):
        batch_indices = np.random.choice(num_samples, size=batch_size, replace=False)
        x_batch = train_data[batch_indices]
        y_batch = labels[batch_indices]

        outputs = model(x_batch)
        loss = criterion(outputs, y_batch)
        grads = torch.autograd.grad(loss, model.parameters())

        with torch.no_grad():
            for p, g in zip(model.parameters(), grads):
                p -= learning_rate * g

        losses.append(loss.item())
        test_accuracies.append(evaluate_model(model, X_test, y_test, batch_size))

        if itr % 100 == 99:
            print(f"Iter {itr+1}, Loss: {loss.item():.4f}, Test Acc: {test_accuracies[-1]:.4f}")

    return losses, test_accuracies

### Learning Rate Experiments
Complete the above functions `train_model` and `evaluate_model` to train the model using Stochastic Gradient Descent and evaluate the model on the test set. Use three different learning rates:
1. 5.0
2. 0.5
3. 0.005

What do you observe? Note that we usually want a learning rate as large as possible such that the loss actually decreases. Which of these learning rates satisfy that criteria?

In [ ]:
model = MLP(28*28, 200, 10)

losses, test_accuracies = train_model(
    model,
    X_train,
    y_train,
    batch_size=100,
    learning_rate=0.005,
    num_iterations=1000
)

plt.plot(losses)
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.grid(True)
plt.show()


plt.plot(test_accuracies)
plt.xlabel("Iteration")
plt.ylabel("Test Accuracy")
plt.title("Test Accuracy")
plt.grid(True)
plt.show()

### Reflection
- Which learning rates diverge?
- Which converge slowly?
- Why is the largest stable learning rate preferred?

### Expected Answers
- LR = 5.0 → diverges (overshoots minima)
- LR = 0.5 → unstable oscillations
- LR = 0.005 → stable but slow

Largest stable LR gives fastest convergence

### Width and Interpolation
Modern neural networks often interpolate the training data.

### Exercise 3
- Vary hidden width: 50, 100, 200, 500, 1000
- Track training loss
- Identify the smallest width where training loss reaches zero

In [ ]:
widths = [50, 100, 200, 500, 1000]
final_losses = []

for w in widths:
    model = MLP(28*28, w, 10)
    losses, _ = train_model(model, X_train, y_train, 100, 0.01, 500)
    final_losses.append(losses[-1])
    print(f"Width {w}, Final Loss {losses[-1]:.6f}")